# DS/CMPSC 410 Spring 2026
## Instructor: Professor John Yen
## TA: Jingxi Zhu and Septarshi Roy

## Lab 4 Data Integration, Change Detection 
## The goals of this lab are for you to be able to
## - Integrate multiple data sources using RDD-based outerjoin
## - Transform None values generated by outerjoin. 
## - Compute temporal changes of values associated with a key.
## - Apply the obove to compute difference of hastag counts of Tweets related to Boston Marathon Bombing between April 17 and April 19.
##  This lab includes four data sets, as explained below.
## Data 
- The first dataset contains Boston Marathon Bombing collected on 4/17/2013. (same as Lab3)
- The second dataset contains Boston Marathon Bombing collected on 4/19/2013. BMB_4_19_tweets.csv is available through cp from /storage/home/juy1/pub)
- Like Lab3, a smaller sampled dataset for each day's tweets has also been provided: sampled_4_17_tweets.csv and sampled_4_19_tweets.csv.  You should use these two data sets for running Spark in the local mode (using Jupyter Notebook).  
- However, you should change input data for spark-submit (cluster mode) to the big dataset for each day: BMB_4_17_tweets.csv and BMB_4_19_tweets.csv.
- Like previous labs, you should create a Lab4 subdirectory under work, because more space is available under work directory. 
- You should also download this Jupyter Notebook for Lab 4 to the same directory.
## Problem
- The problem we want to solve is to (1) find hashtags that in 4/17/2013 tweets and in 4/19/2013 tweets, and (2) calculate the difference of total hashtag counts (i.e., The count for each hashtag on 4/19/2013 - (minus) the count for each hashtag on 4/17/2013). 
- Save these hashtag count difference in text files.
## Two Steps (like Lab3)
- You will first run in Spark local mode (and Jupyter Notebook) for the small sample for each day.
- After you obtain the result for small datasets, you can then convert the code for local mode into code for cluster mode, and submit the code to ICDS cluster and obtain run-time performance.  
1. Export the Jupyter Notebook as Lab4.py file, upload it from your local machine to your Lab4 directory in ICDS.
2. Follow the instructions in "Running Spark in Cluster Mode_S26"
2. Modify the Lab4.py file so that it reads from BMB_4_17_tweets.csv and BMB_4_19_tweets.csv.
4. Modify Lab4.py so that outputs are saved in directories different from those used in the local model
5. Follow the instructions in Lab 3 for "Lab3&Running_Spark_in_Cluster Mode_S26" to modify .py file, request resources, load JDK/Pyspark, and run pbs-spark-submit.

## Submit the following items for Lab 4
- Completed Jupyter Notebook of Lab 4 (50%)
- - Exercise 1: 5%
- - Exercise 2: 15%
- - Exercise 3: 10%
- - Exercise 4: 10%
- - Exercise 5: 10%
- Lab4.py (used for spark-submit) (15%)
- Log file for Lab4.py             (15%)
- A screen shot of "ls -al" in your Lab4 directory (5 %)
- A screen shot of "ls -al" in your cluster output directory for hashtag count difference. (5 %)
- The fist and the second output file of hashtag count difference generated by running spark-submit (cluster mode) on Lab4.py. (10%)

## Total Points: 100 points

# Due: midnight, February 15th (Sunday).

In [1]:
import pyspark

In [2]:
from pyspark import SparkContext

## Because we are not using DataFrame in this lab, we will be creating SparkContext rather than SparkSession

- Note: We use "local" as the master parameter for ``SparkContext`` in this notebook so that we can run and debug it in ICDS Jupyter Server.  However, we need to remove ``"master="local",``later when you convert this notebook into a .py file for running it in the cluster mode.

In [3]:
sc=SparkContext(master="local", appName="Lab4 BMB hastag changes")
sc

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/15 13:42:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/15 13:42:44 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/15 13:42:44 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


<SparkContext master=local appName=Lab4 BMB hastag changes>

In [4]:
sc.setLogLevel("WARN")

# Exercise 1 (5 points)  Add your name below and complete the paths for the two following ``textFile`` statements.
## Answer for Exercise 1
- Your Name: Aarya Soni

## Computing hastag count difference includes three steps:
- Step 1: Compute (and save) hashtag counts for April 17th tweets. 
- Step 2: Compute (and save) hashtag counts for April 19th tweets. 
- Step 3: Combine the hashtag counts of two days, compute their difference, sort based on the difference. Save the hashtag count difference. 

# Step 1 Compute (and save) hashtag counts for April 19th tweets.

## Complete the path and run the code below to read the file "sampled_BMB_4_17_tweets.csv" from your directory.

In [5]:
tweets_D1_RDD = sc.textFile("/storage/home/abs7346/work/Lab3/sampled_BMB_4_17_tweets.csv")
tweets_D1_RDD

/storage/home/abs7346/work/Lab3/sampled_BMB_4_17_tweets.csv MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0


## Execute the code below, which computes the total count of hashtags in the input tweets, sort them by count (in descending order), and save them in an output directory:
- (a) Uses flatMap to "flatten" the list of tokens from each tweet (using split function) into a very large list of tokens.
- (b) Filter the token for hashtags.
- (c) Count the total number of hashtags in a way similar to Lab 2.
- (d) Sort the hashtag count in descending order.
- (e) Save the sorted hashtag counts in an output directory.

In [6]:
tokens_D1_RDD = tweets_D1_RDD.flatMap(lambda line: line.strip().split(" "))

In [7]:
hashtag_D1_RDD = tokens_D1_RDD.filter(lambda x: x.startswith("#"))

In [8]:
hashtag_1_D1_RDD = hashtag_D1_RDD.map(lambda x: (x, 1))

In [9]:
hashtag_count_D1_RDD = hashtag_1_D1_RDD.reduceByKey(lambda x, y: x+y , 10 )

# Step 2 Compute hashtag counts for April 19th tweets.

In [17]:
tweets_D2_RDD = sc.textFile("/storage/home/abs7346/work/Lab4/sampled_BMB_4_19_tweets.csv")
tweets_D2_RDD

/storage/home/abs7346/work/Lab4/sampled_BMB_4_19_tweets.csv MapPartitionsRDD[15] at textFile at NativeMethodAccessorImpl.java:0

## Exercise 2 (10%)
Complete the code below to compute the total count of hashtags in the input tweets, sort them by count (in descending order), and save them in an output directory:
- (a) Uses flatMap to "flatten" the list of tokens from each tweet (using split function) into a very large list of tokens.
- (b) Filter the token for hashtags.
- (c) Count the total number of hashtags in a way similar to Lab 2.
- (d) Sort the hashtag count in descending order.
- (e) Save the sorted hashtag counts in an output directory.

In [18]:
tokens_D2_RDD = tweets_D2_RDD.flatMap(lambda line: line.strip().split(" "))

In [19]:
hashtag_D2_RDD = tokens_D2_RDD.filter(lambda x: x.startswith("#"))

In [20]:
hashtag_1_D2_RDD = hashtag_D2_RDD.map(lambda x: (x, 1) )

In [21]:
hashtag_count_D2_RDD = hashtag_1_D2_RDD.reduceByKey(lambda x, y: x+y , 10 )

# Step 3 Combine the hashcount of two days, compute their difference, and save sorted difference of hashtag counts.

# Exercise 3 (10%) 
## Complete The code below to join the two hashtag-count key value pairs RDDs on their keys (i.e., hashtags), and  compute the difference of the counts between the two days (count of day 2 - count of day 1).

In [22]:
hashtag_count_D1_RDD.take(3)

[('#football', 4), ('#BostonLovee!!!', 1), ('#Caps', 1)]

In [23]:
hashtag_count_D2_RDD.take(3)

[('#boston!', 17), ('#Anonymous', 14), ('#WiseWords', 2)]

In [24]:
joined_hashtag_count_RDD = hashtag_count_D1_RDD.fullOuterJoin(hashtag_count_D2_RDD)

In [25]:
joined_hashtag_count_RDD.take(5)

[('#football', (4, 3)),
 ('#BostonLovee!!!', (1, None)),
 ('#Caps', (1, None)),
 ('#3daysfor117boston', (2, None)),
 ('#united', (21, 4))]

# As we discussed in class, we need to convert missing values (None) into 0 before we calculate the difference of the count.
## Before we do that, let us see whether we can find hashtag whose count (in one of the days) has ``None`` value

In [26]:
none1_RDD = joined_hashtag_count_RDD.filter(lambda pair: pair[1][0]==None )

In [27]:
none1_RDD.take(5)

[('#WiseWords', (None, 2)),
 ('#UTexasPINKPartyâ€\x9d', (None, 3)),
 ('#prayfortexas', (None, 412)),
 ('#1?', (None, 1)),
 ('#itwasntme', (None, 1))]

In [28]:
none1_RDD.count()

8138

In [29]:
none2_RDD= joined_hashtag_count_RDD.filter(lambda pair: pair[1][1]==None)

In [30]:
none2_RDD.take(5)

[('#BostonLovee!!!', (1, None)),
 ('#Caps', (1, None)),
 ('#3daysfor117boston', (2, None)),
 ('#friends', (3, None)),
 ('#SOJC', (1, None))]

In [31]:
none2_RDD.count()

7827

# The following function replaces a missing value (None) with zero.

In [32]:
def tran_none(x):
    if (x==None) :
        return(0)
    else:
        return(x)

# Exercise 4 (10%) 
## Complete the code below to transform missing values of hashtag counts into 0, and CHECK WHETHER THE RESULT IS CORRECT.

In [33]:
tran_joined_hashtag_count_RDD = joined_hashtag_count_RDD.map(lambda x: (x[0], (tran_none(x[1][0]), tran_none(x[1][1]))) )

## Check whether the result does not contain any missing value.  If you still find any missing values in ``tran_joined_hashtag_count_RDD``, double check your answer for Exercise 4.

In [34]:
none3_RDD = tran_joined_hashtag_count_RDD.filter(lambda pair: pair[1][0] == None )

In [35]:
none3_RDD.count()

0

In [36]:
none4_RDD = tran_joined_hashtag_count_RDD.filter(lambda pair: pair[1][1] == None )

In [37]:
none4_RDD.count()

0

# Exercise 5 (15%)
Complete the code below to calculate the difference of hashtag counts between the two days (i.e., hashtag count of 4/19/213 - hashtag count of 4/17/2013), and save the result.

In [40]:
hashtag_count_diff_RDD = tran_joined_hashtag_count_RDD.map(lambda x: (x[0], x[1][1] - x[1][0] ))

In [41]:
hashtag_count_diff_RDD.take(5)

[('#football', -1),
 ('#BostonLovee!!!', -1),
 ('#Caps', -1),
 ('#3daysfor117boston', -2),
 ('#united', -17)]

In [42]:
sorted_hashtag_count_diff_RDD = hashtag_count_diff_RDD.sortBy(lambda pair: pair[1], ascending = False)

In [43]:
sorted_hashtag_count_diff_RDD.take(5)

[('#Boston', 8211),
 ('#boston', 4336),
 ('#Watertown', 3754),
 ('#watertown', 2028),
 ('#manhunt', 1479)]

In [44]:
output_path3 = "/storage/home/abs7346/work/Lab4/sorted_BMB_hashtag_count_diff_local.txt" 
sorted_hashtag_count_diff_RDD.saveAsTextFile(output_path3)

In [45]:
sc.stop()